# Transformer training

Follow the executed training process: data partitions → four aligned views → frame features → Transformer → losses → saved learning curves → selected checkpoint.

The selected model was pretrained on constructed phrases and then adapted using **Vocadito and MTG-Jamendo recordings with synthetic replay**. Known injected shifts define engineering targets. They are not labels of a singer's intended perfect melody.

**Run:** open with `Run-Notebooks.ps1`, choose **BPDR local Python**, then **Restart Kernel and Run All Cells**. In VS Code choose the component `.venv/Scripts/python.exe`. Every cell is safe to run in order: saved training is replayed as evidence; only a CPU forward pass is newly computed. Full fitting is intentionally kept in the training scripts.

Related notebooks: [data](01_input_validation_and_analysis.ipynb), [complete study record](02_data_review_and_paired_experiments.ipynb), [evaluation](04_transformer_evaluation.ipynb).

In [ ]:
# Locate the workspace before importing project modules (JupyterLab or VS Code).
import os, sys, importlib.util
from pathlib import Path
start = Path(os.environ.get('BPDR_COMPONENT_ROOT', Path.cwd())).resolve()
COMPONENT = next((p for p in (start, *start.parents)
    if p.name == 'bpdr_input' and (p/'pyproject.toml').is_file()), None)
if COMPONENT is None:
    raise RuntimeError('Open this notebook inside BPDR_Workspace, using Run-Notebooks.ps1.')
sys.path.insert(0, str(COMPONENT/'BPDR_Workspace/source'))
required = ['numpy', 'pandas', 'matplotlib', 'soundfile', 'torch', 'IPython']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError('Missing packages: '+', '.join(missing)+
        '. Select '+str(COMPONENT/'.venv/Scripts/python.exe')+' as the notebook kernel.')
print('Python:', sys.executable)
print('Workspace:', COMPONENT/'BPDR_Workspace')

In [ ]:
import json, hashlib, inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Audio, Code
from bpdr_input.reporting.notebook_evidence import load_index
from bpdr_input.experiments.evaluation.transfer_study import read, sha
INDEX, EVIDENCE = load_index(COMPONENT)
M = read(EVIDENCE['presentation_manifest'])
assert sha(EVIDENCE['presentation_manifest']) == INDEX['current_gate']['presentation_manifest_sha256']
for path, digest in M['source_hashes'].items():
    assert sha(path) == digest, path
release = EVIDENCE['presentation_manifest'].parent
assert sha(release/'model_card.json') == M['model_card_sha256']
CARD = read(release/'model_card.json')
working = COMPONENT/'BPDR_Workspace/selected_model'
working_card = read(working/'model.json')
CHECKPOINT = working/working_card['checkpoint']
assert sha(CHECKPOINT) == working_card['sha256'] == CARD['checkpoint_sha256']
SYNTH = {seed: read(Path(folder)/'evidence.json') for seed,folder in M['synthetic_runs'].items()}
ADAPT_DIR = EVIDENCE['presentation_adapt'].parent
ADAPT = read(EVIDENCE['presentation_adapt'])
assert sha(ADAPT_DIR/'declaration.json') == ADAPT['declaration_sha256']
print('Verified saved evidence and selected checkpoint:', sha(CHECKPOINT))
print('This notebook reads saved experiments; it does not fit or replace a model.')

## 1. What entered training

Vocadito provides real singing with performed pitch/note references. Jamendo provides separated vocals and predicted musical metadata. Neither automatically supplies a corrected version of an unrestricted novice performance. We construct local detuning/boundary interventions on usable recordings and retain original source identities.

We split by singer, artist proxy or generated voice setting before augmentation. Multiple variants from one phrase are dependent observations, not additional people. The training normalizer is fitted only on training data; adaptation keeps it frozen. Development data is used for evaluation, never optimizer updates or early stopping in this recorded adaptation. Repeated development diagnostics mean it is not an untouched final test.

In [ ]:
pretrain_dir = Path(M['synthetic_runs']['339'])
pretrain_decl = read(pretrain_dir/'declaration.json')
controlled = Path(pretrain_decl['data'])
catalog = read(controlled/'catalog.json')
fine = COMPONENT/'runs/fine_pitch/66e3619e611241dd80d33a32578b73c6'
train_real = read(fine/'training_catalog.json')['records']
dev_real = read(fine/'development_catalog.json')['records']
def dataset(row):
    return row.get('dataset','vocadito' if row['phrase_id'].startswith('vocadito') else 'jamendo')
assert not {(dataset(r),r['singer_id']) for r in train_real} & {(dataset(r),r['singer_id']) for r in dev_real}
rows=[]
for source in ['vocadito','jamendo']:
    for split, records in [('train',train_real),('development',dev_real)]:
        part=[r for r in records if dataset(r)==source]
        rows.append(dict(source=source,split=split,variants=len(part),
            phrases=len({r['phrase_id'] for r in part}),groups=len({r['singer_id'] for r in part})))
for split in ['train','heldout']:
    part=[r for r in catalog['records'] if r['split']==split and r['plateau_frames']>=10]
    rows.append(dict(source='constructed voices',split=split,variants=len(part),
        phrases=len({r['phrase_id'] for r in part}),groups=len({r['voice_id'] for r in part})))
assert not {r['voice_id'] for r in catalog['records'] if r['split']=='train'} & {r['voice_id'] for r in catalog['records'] if r['split']=='heldout'}
display(pd.DataFrame(rows))
print('Verified: no training/development singer or artist-proxy overlap.')

## 2. One training example has four views

| View | Audio | Boundary metadata | Target correction |
| --- | --- | --- | --- |
| A | Original phrase | Nominal | Zero |
| B | Same original | Split / merge / jitter | Zero |
| C | Locally detuned phrase | Nominal | Negative of known added detuning |
| D | Same detuned phrase | Same changed boundaries as B | Same target as C |

The examples below are **actual saved training features**, not a new generated illustration. Pitch is made relative to the phrase's voiced median, while timing-only boundary information occupies the last feature. Supervision/support masks determine where losses and metrics are valid; the model receives acoustic features and boundary metadata, not correction targets or reference audio.

In [ ]:
from bpdr_input.experiments.learning.controlled_study import load_pair
from bpdr_input.experiments.learning.fine_pitch_relative_check import relative_pair
record=next(r for r in catalog['records'] if r['split']=='train' and r['plateau_frames']>=10)
pair=relative_pair(load_pair(record))
fig,axes=plt.subplots(3,1,figsize=(11,8),sharex=True)
for view in 'AC': axes[0].plot(pair['inputs'][view][:,64],label=view,alpha=.8)
for view in 'AB': axes[1].plot(pair['inputs'][view][:,70],label=view,alpha=.8)
axes[2].plot(-pair['detuning'],label='C/D target correction',color='#a44324')
axes[2].fill_between(np.arange(len(pair['mask'])),0,1,where=pair['mask'],
    transform=axes[2].get_xaxis_transform(),alpha=.1,label='Reliable loss mask')
axes[0].set(ylabel='Relative pitch (cents)',title='Saved training example: '+pair['pair_id'])
axes[1].set(ylabel='Boundary score');axes[2].set(xlabel='Aligned frame index',ylabel='Correction (cents)')
for ax in axes: ax.legend(loc='upper right')
plt.tight_layout();plt.show()
display(pd.DataFrame([dict(view=v,frames=pair['inputs'][v].shape[0],features=pair['inputs'][v].shape[1]) for v in 'ABCD']))

## 3. Transformer architecture and a real checkpoint forward pass

Each frame has 64 log-mel bands, fine pitch, confidence, voicing, pitch change, energy and onset strength (**70 acoustic inputs**), plus one timing-only boundary score (**71 total**). Linear projection maps to 128 dimensions; sinusoidal positions preserve ordering. Four bidirectional Transformer encoder layers use four attention heads, feed-forward width 512 and dropout 0.1. A linear head outputs one relative correction in cents per frame. Padding is masked; no causal mask is used.

The frame-only control has 70 inputs; augmentation-only and BPDR have 71. Matched initialization, sampling and budgets make comparisons meaningful. Below we load the checksum-verified **selected adapted checkpoint**, not a random newly initialized model. Inference is CPU-only and leaves its weights unchanged.

In [ ]:
import torch
from bpdr_input.reporting.bpdr_showcase import load_model
from bpdr_input.modeling.examples import collate
from bpdr_input.modeling.model import CorrectionEncoder
torch.set_num_threads(2)
checkpoint_before=sha(CHECKPOINT)
model,normalizer,width=load_model(CHECKPOINT)
model.eval()
batch=collate([pair],normalizer,width)
with torch.inference_mode():
    predictions={view:model(x,batch['padding']) for view,x in batch['inputs'].items()}
display(pd.DataFrame([model.describe()]))
print('Input:',tuple(batch['inputs']['A'].shape),'→ output:',tuple(predictions['A'].shape))
print('Normalization width:',len(normalizer.mean),'| Every output finite:',all(torch.isfinite(p).all().item() for p in predictions.values()))
assert model.parameter_count()==CARD['architecture']['parameters']
assert sha(CHECKPOINT)==checkpoint_before
display(Code(inspect.getsource(CorrectionEncoder.forward),language='python'))

## 4. What the optimizer minimized

Let $r_A,r_B,r_C,r_D$ be corrections, and $e$ the known added detuning. Losses use reliable aligned frames and average per example, then across views.

- **Anchor $L_A$:** masked squared correction error; targets are zero for A/B and $-e$ for C/D. This is the recorded MSE-anchor implementation. Earlier MAE-anchor experiments are retained separately.
- **Smoothness $L_S$:** mean absolute adjacent change of the correction on voiced frames; it does not flatten the sung pitch contour.
- **Boundary invariance $L_B$:** half the masked absolute A–B and C–D prediction differences.
- **Counter-detuning $L_D$:** half the masked absolute errors of $(r_C-r_A)+e$ and $(r_D-r_B)+e$.

Warm-up: $L=L_A+0.1L_S$ for the first 100 updates. BPDR subsequently uses $L=L_A+0.1L_S+L_B+L_D$. Controls keep the anchor and smoothness. Terms have different units (squared cents versus cents); the weighted total is an optimization objective, not an accuracy percentage.

In [ ]:
from bpdr_input.experiments.learning.fine_pitch_mse_check import squared_objective
from bpdr_input.modeling.losses import LossWeights
with torch.inference_mode():
    loss,parts=squared_objective(predictions,batch,'bpdr',LossWeights(),True)
display(pd.DataFrame([{'term':k,'value':float(v)} for k,v in parts.items()]))
print('Selected checkpoint loss on this ONE saved TRAINING example:',float(loss))
print('This forward-pass diagnostic is not a new fit or held-out evaluation.')
display(Code(inspect.getsource(squared_objective),language='python'))

## 5. Exact recorded schedules and training loop

Pretraining: seeds 339/340/341, all three conditions per seed, 1,500 updates per condition, batch 8, AdamW learning rate 0.001, weight decay 0.01, gradient clipping at 1.0; evaluations every 250 updates. Each update draws eight of 384 variants without replacement within the batch. This is sampling, not a literal complete-data epoch traversal. Nominal exposure is 31.25 dataset-equivalents per run.

Adaptation: seed 339, 300 updates per condition, AdamW learning rate 0.0001 and the same clipping/decay. Each batch contains two Vocadito, two Jamendo and four constructed records; real sampling balances phrase groups within each dataset. Every condition starts from its corresponding seed-339 pretrained model. The synthetic normalizer stays frozen. Evaluation uses the final update, with no development early stopping. Nominal real exposure is 1,200 draws over 995 real variants; draws can repeat.

The next cells inspect saved declarations/schedules and display the actual maintained loop. **They do not call `train()` or `adapt()`.** Checkpoints and optimizer histories remain historical evidence.

In [ ]:
adapt_decl=read(ADAPT_DIR/'declaration.json')
schedule=np.load(pretrain_dir/'schedule.npy')
adapt_schedule=np.load(ADAPT_DIR/'schedule.npy')
assert schedule.shape==(1500,8) and adapt_schedule.shape==(300,8)
assert sha(ADAPT_DIR/'schedule.npy')==adapt_decl['schedule_sha256']
display(pd.DataFrame([
    {'stage':'Pretraining per condition/seed','updates':len(schedule),'batch':schedule.shape[1],'learning_rate':.001,'warmup':100,'variants_available':384},
    {'stage':'Real + replay adaptation per condition','updates':len(adapt_schedule),'batch':adapt_schedule.shape[1],'learning_rate':adapt_decl['learning_rate'],'warmup':100,'variants_available':'995 real + 384 replay'}]))
display(pd.DataFrame(adapt_schedule[:5],columns=['Vocadito 1','Vocadito 2','Jamendo 1','Jamendo 2','Replay 1','Replay 2','Replay 3','Replay 4']))
print('Recorded training device:',pretrain_decl['device'])
print('Adaptation normalization:',adapt_decl['normalizer'])
print('Recorded hardware/software:');display(M['training_environment'])
from bpdr_input.experiments.learning import controlled_study
loop=inspect.getsource(controlled_study.train)
display(Code(loop[loop.index('    for condition, width'):loop.index("    (out/'evidence.json')")],language='python'))

## 6. Saved learning curves, including unstable runs

These points are read from completed training receipts. Pretraining loss is the logged last sampled batch at each evaluation interval; it is not an epoch mean. The accompanying evaluation error/response is on excluded constructed voice settings. Those settings are not independent human singers. Displaying every seed prevents a favorable run from hiding instability.

In [ ]:
fig,axes=plt.subplots(3,3,figsize=(14,10),sharex=True)
for row,seed in enumerate(['339','340','341']):
    for condition,result in SYNTH[seed]['results'].items():
        curve=result['curve'];steps=[p['update'] for p in curve]
        axes[row,0].plot(steps,[p['loss'] for p in curve],label=condition)
        axes[row,1].plot(steps,[p['heldout']['correction_mae'] for p in curve],label=condition)
        axes[row,2].plot(steps,[p['heldout']['median_response_gain'] for p in curve],label=condition)
    axes[row,0].set(ylabel='Seed '+seed,title='Sampled training objective')
    axes[row,1].set(title='Excluded-voice MAE (cents)');axes[row,2].set(title='Paired response gain')
    axes[row,2].axhline(1,color='grey',ls=':')
for ax in axes[-1]:ax.set_xlabel('Optimizer update')
axes[0,0].legend(fontsize=8);plt.tight_layout();plt.show()

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
for condition in ['augmentation_only','frame_only','bpdr']:
    history=read(ADAPT_DIR/(condition+'_history.json'))
    assert len(history)==300
    frame=pd.DataFrame(history)
    axes[0].plot(frame['update'],frame['loss'].rolling(15,min_periods=1).mean(),label=condition)
    axes[1].plot(frame['update'],frame['gradient_norm'].rolling(15,min_periods=1).mean(),label=condition)
axes[0].set(title='Adaptation objective: 15-update moving mean',xlabel='Update',ylabel='Weighted loss')
axes[1].set(title='Gradient norm before clipping: moving mean',xlabel='Update',ylabel='Norm')
axes[0].legend(fontsize=8);plt.tight_layout();plt.show()
print('Lower training loss alone does not establish generalization; evaluation follows in notebook 04.')

## 7. What was selected, and what remains open

The working checkpoint is the seed-339 BPDR model after real-data adaptation with replay. It is retained for an experimental prototype, not adopted as a validated general real-singer repair model. The later A4-safe pilot and real-humming adaptation did not pass their declared evaluation checks and did not replace this checkpoint.

To reproduce fitting on capable hardware, inspect the recorded declaration, schedule, archived executed source and environment before running the training functions. The source modules are `experiments/learning/controlled_study.py` and `experiments/evaluation/transfer_study.py`. Their `train` / `adapt` commands are deliberately absent from executable notebook cells because rerunning them produces new models and must be treated as a new experiment.

Continue to [evaluation](04_transformer_evaluation.ipynb) for matched controls, all seeds, real-source results, uncertainty, preservation and audible examples.

In [ ]:
display(pd.DataFrame([{'checkpoint':str(CHECKPOINT),'sha256':sha(CHECKPOINT),
    'training':working_card['training'],'research_adopted':working_card['research_model_adopted'],
    'prepared_handoff_enabled':working_card['prepared_handoff_enabled']}]))
assert sha(CHECKPOINT)==checkpoint_before
print('Completed training walkthrough. No optimizer updates were executed.')